# Best Practices for Integration Testing

## Overview

Integration testing is about verifying that different modules work together correctly. Here are best practices to ensure your integration tests are effective and maintainable.

---

## 1. Test Real Interactions, Not Just Return Values

### ❌ Bad Practice
```python
def test_add_task():
    service = Service()
    result = service.add_task("Task")
    assert result is True  # Only checks return value
```

### ✅ Good Practice
```python
def test_add_task():
    service = Service()
    result = service.add_task("Task")
    
    # Verify return value
    assert result is True
    
    # Verify side effects
    tasks = service.get_tasks()
    assert len(tasks) == 1
    assert tasks[0]['title'] == "Task"
    
    # Verify notifications
    notifications = service.get_notifications()
    assert len(notifications) == 1
```

## 2. Use Appropriate Test Doubles

### When to Use Stubs
- When testing higher-level modules before lower-level ones are ready
- When you need to simulate specific response scenarios
- When dependencies are slow or unreliable

### When to Use Mocks
- When you need to verify that methods were called
- When you need to verify call arguments
- When you need to control the behavior of dependencies

### When to Use Real Components
- When testing critical integrations
- When the component is fast and reliable
- When you need to test the actual implementation

In [ ]:
from unittest.mock import Mock

# Example: Using mocks to verify interactions
def test_add_task_with_mocks():
    storage_mock = Mock()
    notifier_mock = Mock()
    service = Service(storage=storage_mock, notifier=notifier_mock)
    
    result = service.add_task("Test Task", "Description")
    
    # Verify the interactions
    storage_mock.save.assert_called_once()
    notifier_mock.send.assert_called_once_with("New task added: Test Task")
    
    print("Mock-based test passed!")

## 3. Test Error Scenarios

Integration tests should cover not just happy paths, but also error scenarios:

- What happens when storage fails?
- What happens when notification fails?
- What happens when both fail?
- What happens with invalid input?

In [ ]:
def test_storage_failure():
    """Test behavior when storage fails."""
    storage_mock = Mock()
    storage_mock.save.side_effect = Exception("Storage failed")
    notifier_mock = Mock()
    
    service = Service(storage=storage_mock, notifier=notifier_mock)
    
    # Should raise the exception
    try:
        service.add_task("Test Task")
        assert False, "Should have raised exception"
    except Exception as e:
        assert str(e) == "Storage failed"
    
    # Notifier should not have been called
    notifier_mock.send.assert_not_called()
    
    print("Storage failure test passed!")

def test_notifier_failure():
    """Test behavior when notifier fails."""
    storage_mock = Mock()
    notifier_mock = Mock()
    notifier_mock.send.side_effect = Exception("Notifier failed")
    
    service = Service(storage=storage_mock, notifier=notifier_mock)
    
    # Should still succeed (notification failure is handled)
    result = service.add_task("Test Task")
    assert result is True
    
    # Storage should have been called
    storage_mock.save.assert_called_once()
    
    print("Notifier failure test passed!")

## 4. Test Edge Cases

Don't forget to test edge cases in integration:

- Empty strings
- None values
- Very long strings
- Special characters
- Duplicate data

In [ ]:
def test_empty_title():
    """Test that empty titles are rejected."""
    service = Service()
    
    try:
        service.add_task("")
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert "Title cannot be empty" in str(e)
    
    print("Empty title test passed!")

def test_whitespace_title():
    """Test that whitespace-only titles are rejected."""
    service = Service()
    
    try:
        service.add_task("   ")
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert "Title cannot be empty" in str(e)
    
    print("Whitespace title test passed!")

## 5. Keep Tests Independent

Each test should be independent and not rely on the state left by previous tests.

### ❌ Bad Practice
```python
def test_add_first_task():
    service = Service()
    service.add_task("Task 1")
    assert len(service.get_tasks()) == 1

def test_add_second_task():
    service = Service()  # Reuses same instance!
    service.add_task("Task 2")
    assert len(service.get_tasks()) == 2  # Depends on previous test
```

### ✅ Good Practice
```python
def test_add_first_task():
    service = Service()  # Fresh instance
    service.add_task("Task 1")
    assert len(service.get_tasks()) == 1

def test_add_second_task():
    service = Service()  # Fresh instance
    service.add_task("Task 1")
    service.add_task("Task 2")
    assert len(service.get_tasks()) == 2
```

## 6. Use Descriptive Test Names

Test names should clearly describe what they test and what the expected outcome is.

### ❌ Bad Practice
```python
def test_1():
def test_service():
def test_add():
```

### ✅ Good Practice
```python
def test_add_task_saves_to_storage_and_sends_notification():
def test_add_task_with_empty_title_raises_value_error():
def test_add_task_with_duplicate_title_raises_value_error():
```

## 7. Test Consistency Across Failures

When one component fails, verify that the system remains in a consistent state.

### Example: Partial Failure Handling

If storage succeeds but notification fails:
- The task should still be saved
- The system should not be in an inconsistent state
- The operation should still be considered successful (if notification is non-critical)

In [ ]:
def test_partial_failure_consistency():
    """Test that system remains consistent when notification fails."""
    storage = Storage()
    notifier_mock = Mock()
    notifier_mock.send.side_effect = Exception("Notification failed")
    
    service = Service(storage=storage, notifier=notifier_mock)
    
    # Should still succeed (notification is non-critical)
    result = service.add_task("Test Task")
    assert result is True
    
    # Task should be saved despite notification failure
    tasks = storage.get_all()
    assert len(tasks) == 1
    assert tasks[0]['title'] == "Test Task"
    
    print("Partial failure consistency test passed!")

## 8. Organize Tests by Feature

Group related tests together using test classes or modules.

### Example Structure
```
tests/
├── test_service.py
│   ├── TestServiceAddTask
│   ├── TestServiceGetTasks
│   └── TestServiceErrorHandling
├── test_storage.py
│   ├── TestStorageSave
│   └── TestStorageGetAll
└── test_notifier.py
    └── TestNotifierSend
```

## 9. Use Fixtures for Common Setup

Pytest fixtures help reduce code duplication and make tests more maintainable.

In [ ]:
import pytest

@pytest.fixture
def service():
    """Fixture that provides a fresh Service instance."""
    return Service()

@pytest.fixture
def service_with_mocks():
    """Fixture that provides a Service with mocked dependencies."""
    storage_mock = Mock()
    notifier_mock = Mock()
    return Service(storage=storage_mock, notifier=notifier_mock)

# Tests using fixtures
def test_add_task_with_fixture(service):
    result = service.add_task("Test Task")
    assert result is True

def test_add_task_with_mocks_fixture(service_with_mocks):
    result = service_with_mocks.add_task("Test Task")
    assert result is True
    service_with_mocks.storage.save.assert_called_once()
    service_with_mocks.notifier.send.assert_called_once()

print("Fixture examples defined!")

## 10. Document Complex Scenarios

For complex integration scenarios, add comments or docstrings explaining the test purpose.

In [ ]:
def test_storage_failure_prevents_notification():
    """
    Test that when storage fails, notification is not sent.
    
    This is important because:
    - We don't want to notify about tasks that weren't saved
    - It maintains system consistency
    - It prevents false user notifications
    """
    storage_mock = Mock()
    storage_mock.save.side_effect = Exception("Storage failed")
    notifier_mock = Mock()
    
    service = Service(storage=storage_mock, notifier=notifier_mock)
    
    try:
        service.add_task("Test Task")
        assert False, "Should have raised exception"
    except Exception:
        pass
    
    # Verify notification was not sent
    notifier_mock.send.assert_not_called()
    
    print("Complex scenario test passed!")

## Summary Checklist

Before considering your integration tests complete, verify:

- [ ] Tests verify real interactions, not just return values
- [ ] Appropriate test doubles are used (stubs, mocks, real components)
- [ ] Error scenarios are covered
- [ ] Edge cases are tested
- [ ] Tests are independent
- [ ] Test names are descriptive
- [ ] System consistency is verified across failures
- [ ] Tests are organized by feature
- [ ] Fixtures are used for common setup
- [ ] Complex scenarios are documented

---

## Final Thought

**Good integration tests are like a safety net for your system. They catch issues that unit tests miss and ensure that your components work together as intended.**